# معمل إنشاء الأدوات المخصصة (`@tool`) وحلقة التنفيذ اليدوية (`bind_tools`)

أهلاً بك في المعمل العملي للمحاضرة 61!

في المحاضرات السابقة شفنا دالة `create_agent` الجاهزة، بس في المعمل ده هننزل خطوة تحت الغطاء عشان نفهم **السر الحقيقي وراء عمل الأدوات**:
1. إزاي نحول أي دالة بايثون عادية لأداة ذكية باستخدام `@tool`.
2. إزاي نربطها بالموديل عن طريق `model.bind_tools()`.
3. إزاي بنبني **حلقة التنفيذ اليدوية (`Tool Execution Loop`)** خطوة بخطوة بيدينا!


In [1]:
# تحميل الموديل وتجهيز مكتبة الأدوات
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_groq import ChatGroq

load_dotenv()
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("تم تجهيز الموديل بنجاح.")


تم تجهيز الموديل بنجاح.


### 1. إنشاء الأداة وفحص مخططها (`Schema Introspection`)

بمجرد ما تحط `@tool` فوق الدالة، LangChain بتقرا اسم الدالة والـ `Docstring` والأنواع، وتبني منهم وصف كامل للموديل.


In [2]:
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

print("اسم الأداة:", get_weather.name)
print("وصف الأداة:", get_weather.description)
print("المعاملات المطلوبة:", get_weather.args)


اسم الأداة: get_weather
وصف الأداة: Get the weather at a location.
المعاملات المطلوبة: {'location': {'title': 'Location', 'type': 'string'}}


### 2. ربط الأداة بالموديل (`model.bind_tools`)

ركز جداً هنا! `bind_tools` ما بتشغلش الدالة؛ الموديل بس بيقترح عليك إنك تشغلها من خلال حقل اسمه `tool_calls`. تعال نشوف ده بعينينا!


In [3]:
model_with_tools = model.bind_tools([get_weather])

messages = [{"role": "user", "content": "What's the weather like in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

print("هل الموديل رجع إجابة نصية؟", repr(ai_msg.content))
print("طلب تشغيل الأداة اللي اقترحه الموديل:", ai_msg.tool_calls)


هل الموديل رجع إجابة نصية؟ ''
طلب تشغيل الأداة اللي اقترحه الموديل: [{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_7bdd3ceb-9c73-4664-af86-acfd2f0dbfc2', 'type': 'tool_call'}]


### 3. تشغيل حلقة التنفيذ اليدوية (`Tool Execution Loop`)

دلوقتي دور كود البايثون بتاعنا:
1. ناخد كل طلب من `ai_msg.tool_calls`.
2. ننفذ الدالة بـ `get_weather.invoke(tool_call)` اللي بترجع رسالة `ToolMessage` جاهزة ومربوطة بـ ID الطلب!
3. نبعت سجل الرسائل الكامل للموديل تاني عشان يصيغ الرد النهائي.


In [4]:
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    print("نتيجة الأداة (ToolMessage):", tool_result.content)
    messages.append(tool_result)

final_answer = model_with_tools.invoke(messages)
print("\nالرد النهائي من الموديل:", final_answer.content)


نتيجة الأداة (ToolMessage): It's sunny in Boston.



الرد النهائي من الموديل: It’s sunny in Boston today—great weather for a walk or a day out! If you need more details (temperature, wind, humidity), just let me know.


### 4. جرب بنفسك!

دلوقتي دورك للتجربة الحرة:
1. اكتب أداة جديدة خاصة بيك بـ `@tool` (مثلاً دالة تضرب رقمين `multiply_numbers(a: int, b: int)` أو تحول الدولار للجنيه).
2. اربطها بالموديل في قائمة `bind_tools([get_weather, your_tool])`.
3. اسأل سؤالاً وشوف الموديل بيختار أداتك إزاي ويطلب تشغيلها في `tool_calls`!


In [5]:
# جرب بنفسك هنا: أنشئ أداة جديدة واربطها بالموديل
@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two integers together."""
    return a * b

my_model = model.bind_tools([get_weather, multiply_numbers])
test_msg = my_model.invoke("What is 12 multiplied by 8?")

print("الأداة المختارة:", test_msg.tool_calls[0]["name"])
print("القيم المستخرجة:", test_msg.tool_calls[0]["args"])
print("ناتج تنفيذ الأداة:", multiply_numbers.invoke(test_msg.tool_calls[0]).content)


الأداة المختارة: multiply_numbers
القيم المستخرجة: {'a': 12, 'b': 8}
ناتج تنفيذ الأداة: 96
